# Code Cell 1

In [ ]:
!pip install pyserial pyautogui pillow

# Code Cell 2

In [ ]:
import serial
import time
import pyautogui
from datetime import datetime
import os

# -------- CONFIG --------
PORT = "COM3"     # Change to your Arduino port (e.g., /dev/ttyUSB0 for Linux)
BAUD = 9600
SAVE_DIR = "screenshots"
SCREEN_REGION = None  # set to (x, y, width, height) if you only want a part
# ------------------------

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

print(f"Connecting to Arduino on {PORT}...")
ser = serial.Serial(PORT, BAUD, timeout=1)
time.sleep(2)
print("Connected! Waiting for CAPTURE signal...\n")

def take_screenshot():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(SAVE_DIR, f"screenshot_{timestamp}.png")
    img = pyautogui.screenshot(region=SCREEN_REGION)
    img.save(filename)
    print(f"�� Screenshot saved: {filename}")

try:
    while True:
        if ser.in_waiting:
            line = ser.readline().decode(errors='ignore').strip()
            if line:
                print(f"[Arduino] {line}")
                if line == "CAPTURE":
                    take_screenshot()

except KeyboardInterrupt:
    print("\nStopped manually.")
finally:
    ser.close()
    print("Serial closed.")

# Code Cell 3

In [ ]:
def take_screenshot():
    SAVE_DIR = "./screenshots"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(SAVE_DIR, f"screenshot_{timestamp}.png")
    img = pyautogui.screenshot(region=SCREEN_REGION)
    img.save(filename)
    print(f"�� Screenshot saved: {filename}")

# Code Cell 4

In [ ]:
import serial
import time
import pyautogui
from datetime import datetime
import os

# -------- CONFIG --------
PORT = "COM3"     # Change to your Arduino port (e.g., /dev/ttyUSB0 for Linux)
BAUD = 9600

# Click coordinate (x, y) in screen pixels.
# Example: (400, 300)
CLICK_COORD = (370, 1550)

# Optional: if you want a percent-based coordinate instead of absolute pixels,
# set USE_PERCENT = True and provide (percent_x, percent_y) where 50 means 50%.
USE_PERCENT = False
PERCENT_COORD = (50, 80)

# small delay before click to let UI settle (seconds)
CLICK_DELAY = 0.12
# ------------------------

print(f"Connecting to Arduino on {PORT}...")
ser = serial.Serial(PORT, BAUD, timeout=1)
time.sleep(2)
print("Connected! Waiting for CAPTURE signal...\n")

def perform_click():
    # determine final x,y
    screenW, screenH = pyautogui.size()
    if USE_PERCENT:
        px, py = PERCENT_COORD
        x = int((px / 100.0) * screenW)
        y = int((py / 100.0) * screenH)
    else:
        x, y = CLICK_COORD

    # clamp to valid screen area
    x = max(0, min(screenW - 1, int(x)))
    y = max(0, min(screenH - 1, int(y)))

    print(f"Waiting {CLICK_DELAY}s then clicking at ({x}, {y}) on screen {screenW}x{screenH}...")
    time.sleep(CLICK_DELAY)

    try:
        pyautogui.click(x, y)
        print(f"✅ Clicked at {x},{y}")
    except Exception as e:
        print("❌ Click failed:", e)

try:
    while True:
        if ser.in_waiting:
            line = ser.readline().decode(errors='ignore').strip()
            if line:
                print(f"[Arduino] {line}")
                # Accept either exact CAPTURE or lowercase variants just in case
                if line.upper() == "CAPTURE":
                    perform_click()

except KeyboardInterrupt:
    print("\nStopped manually.")
finally:
    ser.close()
    print("Serial closed.")